# Crossed Design and Response Similarity

::::::::{div}
:class: progress-analysis-app progress4-app

:::::::{div}
:class: progress2-section progress4-input

## Agenda

### Reframe the current regression as ANOVA

Use the [current regression in Progress 3](Progress-3.ipynb#regression-equation) as the starting point. Express the analysis in terms of prompt effects, temperature effects and their interaction, then define the corresponding ANOVA tests.

The current regression already treats prompt and temperature as categorical factors and includes their interaction. A factorial ANOVA can express the same linear model. The next step is to make the hypotheses and variance decomposition explicit.

### Compare responses across prompts using a crossed design

Let $P_1$ and $P_2$ be two prompt versions for the same question. Let $R_{ij}$ denote response $j$ generated from prompt $P_i$, and let $F(P_i,R_{kj})$ evaluate a response against prompt $P_i$. Cross the evaluation prompt with the source of the response: evaluate an answer generated from $P_2$ against $P_1$, then compare it with an answer generated from $P_1$.

::::::{admonition} Worked crossed design example
:class: tip ds-example-card

Generate repeated answers from each prompt:

$$
P_1 \longrightarrow R_{11},R_{12},R_{13},
\qquad
P_2 \longrightarrow R_{21},R_{22},R_{23}.
$$

Keep $P_1$ as the evaluation prompt and compare:

$$
F(P_1,R_{11})
\qquad\text{and}\qquad
F(P_1,R_{21}).
$$

The second score evaluates an answer generated from $P_2$ against $P_1$. How much does that score differ from the score for $R_{11}$?

Also compare the distance between two answers generated from $P_1$ with the distance between answers generated from different prompts:

$$
D(R_{11},R_{12})
\qquad\text{and}\qquad
D(R_{11},R_{21}).
$$

Does changing the prompt produce a larger difference than generating another answer from the same prompt? Extend the evaluation to $F(P_i,R_{kj})$ across evaluation prompts and response sources.
::::::

### Decide what to measure and connect it to ANOVA

Decide what we want to measure: the evaluation score of each answer, the difference between answers, or both. How would each measure connect to ANOVA? Use a few responses from $P_1$ and $P_2$ to make the comparison concrete.

### Find suitable modelling for scores bounded by 0 and 1

Research which statistical models are appropriate when scores lie in $[0,1]$ and the ideal score is 1. The aim is to model the bounded score and its relationship to prompt and response source, not to make the outcome follow a normal distribution.

Identify approaches that can represent scores near the upper limit and scores exactly equal to 1.

### Define text distance within the crossed design

Define whether distance should reflect wording, meaning or disagreement in factual content. Research semantic similarity and syntactic similarity as ways to compare responses. Look into sentiment analysis as a way to compare agreement and disagreement between answers.

Connect these methods to $D(R_{11},R_{12})$ and $D(R_{11},R_{21})$. Determine which measure can distinguish variation between repeated answers to the same prompt from changes associated with a different prompt. Relate these distances to the comparison between $F(P_1,R_{11})$ and $F(P_1,R_{21})$.

:::::::

:::::::{div}
:class: progress2-section progress4-anova

## The current model as ANOVA

The [Progress 3 regression](Progress-3.ipynb#regression-equation) already contains the required factors. For response $j$ from prompt $p$ at temperature $t$, let $\bar Y_{ptj}$ be its average over three evaluations:

$$
\bar Y_{ptj}=\mu+\alpha_p+\tau_t+(\alpha\tau)_{pt}+\varepsilon_{ptj}.
$$

Prompt has four levels and temperature has three. The prompt test compares means averaged equally over temperatures; the temperature test averages equally over prompts. The interaction test asks whether prompt differences change with temperature. Faithfulness and Answer Relevancy are fitted separately.

| Metric | Effect | Degrees of freedom | F | p |
| --- | --- | --- | --- | --- |
| Faithfulness | Prompt main effect | 3, 23 | 6.645 | 0.0021 |
| Faithfulness | Temperature main effect | 2, 23 | 1.168 | 0.3288 |
| Faithfulness | Prompt × Temperature interaction | 6, 23 | 1.057 | 0.4160 |
| Answer Relevancy | Prompt main effect | 3, 23 | 1.357 | 0.2806 |
| Answer Relevancy | Temperature main effect | 2, 23 | 3.418 | 0.0502 |
| Answer Relevancy | Prompt × Temperature interaction | 6, 23 | 0.294 | 0.9334 |

Each analysis contains 35 response averages. Original at temperature 0.6, response 2, lacks a complete evaluation triplet in the saved Progress 3 run. The unit is a generated response, not each evaluation.

#### Conclusion

* **Faithfulness:** the prompt test gives $p=0.0021$, but this result depends strongly on the three “Yes” answers scored zero. Removing them changes it to $p=0.0545$.
* **Answer Relevancy:** prompt differences are not clearly supported. The temperature result is borderline ($p=0.05017$); neither metric shows clear interaction evidence.

:::::{dropdown} Hypotheses, sums of squares and assumptions

The tests use Type III sums of squares with sum contrasts. With an interaction present, main effects refer to equal weighted marginal means. These are different hypotheses from the individual reference level coefficients in Progress 3.

$$
H_P:\alpha_1=\cdots=\alpha_4=0,\qquad
H_T:\tau_1=\tau_2=\tau_3=0,\qquad
H_{PT}:(\alpha\tau)_{pt}=0\ \text{for all }p,t.
$$

For each tested term, $F=(SS_{\mathrm{term}}/df_{\mathrm{term}})/(SSE/23)$. The total decomposition is:

| Metric | Total SS | Model SS | Residual SS |
| --- | --- | --- | --- |
| Faithfulness | 1.38627 | 0.77294 | 0.61333 |
| Answer Relevancy | 0.81601 | 0.29746 | 0.51855 |

The three adjusted Type III term sums need not add to the model SS because one cell has a missing response. Recoding the regression changes its coefficients, but its fitted values agree to numerical precision.

The classical F tests assume independent response errors with common variance and an approximately Gaussian error model. With only two or three responses per condition, these are exploratory working assumptions, not established properties of the data. The tests are not adjusted for multiple comparisons.

The three zero Faithfulness scores came from the judge treating “Yes” as a claim about oral argument recordings. That interpretation needs review. The exclusion above is a sensitivity check, not a replacement analysis. All results concern one underlying question.
:::::

:::::::

:::::::{div}
:class: progress2-section progress4-crossed

## A full crossed evaluation experiment

All four response sources are now crossed with all four evaluation prompts, separately at temperatures 0.3, 0.6 and 0.9. The 36 saved Qwen answers each receive three fresh evaluations under every prompt:

$$
\begin{aligned}
N_{\mathrm{answers}} &= 4\times3\times3 = 36,\\[6pt]
N_{\mathrm{evaluations}} &= 36\times4\times3 = 432.
\end{aligned}
$$

Let $R_{ktj}$ be answer $j$ generated from prompt $P_k$ at temperature $t$. Score $F(P_i,R_{ktj})$ evaluates that fixed answer against prompt $P_i$. Rows below change the response source; columns change the evaluation prompt. The diagonal evaluates answers with their own generation prompt. Every off diagonal cell is a crossed combination.

**417 of 432 evaluations passed validation.** There are 139 complete sets of three evaluations out of 144 planned answer and evaluation prompt combinations. A failed evaluation stays missing; it is never assigned a zero.

### Scores by response source and evaluation prompt

Each cell gives the mean of the available response averages. Parentheses show the number of complete answers out of three. Each response average requires all three evaluations to pass. Where counts differ, differences between cells may also reflect which answers are missing.

### Temperature 0.3

| Response source | Evaluate with Original | Evaluate with A | Evaluate with B | Evaluate with C |
| --- | --- | --- | --- | --- |
| Original | 0.767 (3/3) | 0.833 (3/3) | 0.767 (3/3) | 0.889 (3/3) |
| A | 0.963 (3/3) | 0.907 (3/3) | 0.915 (3/3) | 0.963 (3/3) |
| B | 0.841 (3/3) | 0.775 (3/3) | 0.775 (3/3) | 0.841 (3/3) |
| C | 0.944 (3/3) | 0.822 (3/3) | 0.811 (3/3) | 0.822 (3/3) |

### Temperature 0.6

| Response source | Evaluate with Original | Evaluate with A | Evaluate with B | Evaluate with C |
| --- | --- | --- | --- | --- |
| Original | 0.750 (2/3) | 0.750 (2/3) | 0.750 (2/3) | 0.750 (2/3) |
| A | 0.751 (3/3) | 0.818 (3/3) | 0.677 (3/3) | 0.818 (3/3) |
| B | 0.667 (3/3) | 0.667 (3/3) | 0.667 (3/3) | 0.667 (3/3) |
| C | 0.771 (3/3) | 0.771 (3/3) | 0.771 (3/3) | 0.771 (3/3) |

### Temperature 0.9

| Response source | Evaluate with Original | Evaluate with A | Evaluate with B | Evaluate with C |
| --- | --- | --- | --- | --- |
| Original | 0.679 (3/3) | 0.735 (3/3) | 0.735 (3/3) | 0.735 (3/3) |
| A | 0.740 (3/3) | 0.824 (2/3) | 0.754 (3/3) | 0.787 (3/3) |
| B | 0.708 (3/3) | 0.708 (3/3) | 0.708 (3/3) | 0.750 (3/3) |
| C | 0.843 (3/3) | 0.884 (3/3) | 0.843 (3/3) | 0.843 (3/3) |

### Conclusion

* 15 of 34 complete answers change score under at least one evaluation prompt.
* All 432 planned evaluations were attempted: 417 passed validation and 15 failed. Missing answers limit comparisons in the affected cells.
* The experiment still contains one underlying question. It tests these four prompt versions and saved answers, rather than establishing general performance across questions.

:::::{dropdown} All 36 response score rows

Each entry is an average of three valid evaluations. “Missing” means that at least one evaluation failed. Different prompt columns use the exact same response text.

| Temperature | Source | Response | Original | A | B | C |
| --- | --- | --- | --- | --- | --- | --- |
| 0.3 | A | 1 | 1.000 | 0.833 | 1.000 | 1.000 |
| 0.3 | A | 2 | 1.000 | 1.000 | 0.857 | 1.000 |
| 0.3 | A | 3 | 0.889 | 0.889 | 0.889 | 0.889 |
| 0.3 | B | 1 | 1.000 | 0.800 | 0.800 | 1.000 |
| 0.3 | B | 2 | 0.667 | 0.667 | 0.667 | 0.667 |
| 0.3 | B | 3 | 0.857 | 0.857 | 0.857 | 0.857 |
| 0.3 | C | 1 | 0.833 | 0.667 | 0.833 | 0.667 |
| 0.3 | C | 2 | 1.000 | 0.800 | 0.800 | 0.800 |
| 0.3 | C | 3 | 1.000 | 1.000 | 0.800 | 1.000 |
| 0.3 | Original | 1 | 0.800 | 1.000 | 0.800 | 1.000 |
| 0.3 | Original | 2 | 0.667 | 0.667 | 0.667 | 0.667 |
| 0.3 | Original | 3 | 0.833 | 0.833 | 0.833 | 1.000 |
| 0.6 | A | 1 | 0.600 | 0.800 | 0.600 | 0.800 |
| 0.6 | A | 2 | 0.875 | 0.875 | 0.875 | 0.875 |
| 0.6 | A | 3 | 0.778 | 0.778 | 0.556 | 0.778 |
| 0.6 | B | 1 | 0.500 | 0.500 | 0.500 | 0.500 |
| 0.6 | B | 2 | 1.000 | 1.000 | 1.000 | 1.000 |
| 0.6 | B | 3 | 0.500 | 0.500 | 0.500 | 0.500 |
| 0.6 | C | 1 | 0.800 | 0.800 | 0.800 | 0.800 |
| 0.6 | C | 2 | 0.714 | 0.714 | 0.714 | 0.714 |
| 0.6 | C | 3 | 0.800 | 0.800 | 0.800 | 0.800 |
| 0.6 | Original | 1 | 0.750 | 0.750 | 0.750 | 0.750 |
| 0.6 | Original | 2 | 0.750 | 0.750 | 0.750 | 0.750 |
| 0.6 | Original | 3 | Missing | Missing | Missing | Missing |
| 0.9 | A | 1 | 0.947 | 0.947 | 0.947 | 0.947 |
| 0.9 | A | 2 | 0.571 | Missing | 0.714 | 0.714 |
| 0.9 | A | 3 | 0.700 | 0.700 | 0.600 | 0.700 |
| 0.9 | B | 1 | 0.875 | 0.750 | 0.750 | 0.875 |
| 0.9 | B | 2 | 0.750 | 0.875 | 0.875 | 0.875 |
| 0.9 | B | 3 | 0.500 | 0.500 | 0.500 | 0.500 |
| 0.9 | C | 1 | 1.000 | 1.000 | 1.000 | 1.000 |
| 0.9 | C | 2 | 0.778 | 0.778 | 0.778 | 0.778 |
| 0.9 | C | 3 | 0.750 | 0.875 | 0.750 | 0.750 |
| 0.9 | Original | 1 | 0.571 | 0.571 | 0.571 | 0.571 |
| 0.9 | Original | 2 | 0.667 | 0.833 | 0.833 | 0.833 |
| 0.9 | Original | 3 | 0.800 | 0.800 | 0.800 | 0.800 |
:::::

:::::{dropdown} Failed evaluations

| Failure | Evaluations |
| --- | --- |
| AssertionError: 12 statements but 11 verdicts | 12 |
| AssertionError: 7 statements but 8 verdicts | 3 |

A statement count mismatch means that the judge returned a different number of verdicts from extracted statements. Such a score has an ambiguous denominator and is excluded. Raw attempts and error details are retained, with no retries chosen according to the resulting score.
:::::

:::::{dropdown} Prompts, metric and settings

**Original:** I need to do some legal research to be prepared for my oral argument. Can I visit the law library?

**A:** I'm preparing for an oral argument and need to conduct some legal research. Can I access the law library?

**B:** To be adequately prepared for my oral argument, I require some legal research. Is it possible to visit the law library?

**C:** I'm getting ready for an oral argument and need to do some research. Can I go to the law library?

The metric is DeepEval Answer Relevancy, which evaluates an answer against its input question. The current Faithfulness implementation uses the answer and retrieved context, so changing the evaluation prompt would not change its input.

All 432 attempts are fresh. The earlier Original and B pilot is retained separately and is not reused in these averages. Generation uses the saved Qwen3:4b answers. Evaluation uses local Llama 3.1 8B with temperature 0, context length 8192 and output limit 4096, through DeepEval 4.3.1. Each evaluation uses a fresh conversation. The fixed run order cycles evaluation prompts within answer, then answers within evaluation repeat.
:::::


:::::::

::::::::
